One important thing to do before we start is to get an idea of how many files we will have to deal with. Depending on the number, it could influence our strategy of what we look at, as well as help inform us about the amount of storage we will use etc.

The files we are interested in are visibility files of our 41 stars that contain at least 1 hour of observing.

In [97]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.table import Column
import tqdm
from astropy.table import join
from astroquery.simbad import Simbad
from ipyaladin import Aladin
from astroquery.casda import Casda
from astropy.coordinates import SkyCoord
from astroquery.utils.tap.core import TapPlus
import astropy.units as u
import anywidget
import ipywidgets
import ipyaladin

In [2]:
OPAL_USER = 'jfer9940@uni.sydney.edu.au'
Casda.login(username=OPAL_USER, store_password=True)

INFO: Authenticating jfer9940@uni.sydney.edu.au on CASDA ... [astroquery.casda.core]
INFO: Authentication successful! [astroquery.casda.core]


The first step is to create a list of coordinates for our sources so that we can query CASDA, since CASDA doesn't accept source names in its searches.

Note there were 4 missing sources on first pass. When we investigate a little bit, we see they are in the SRSC, they just have an extra space in their main_id name that we need to account for, so let's edit these cells first before we do our coordinate match.

In [5]:
solar_like_stars = Table.read('data/simbad_classifications.csv')

In [29]:
solar_like_stars

main_id,sp_type,otype,notes
str19,str8,str3,str43
HD 175897,G5V,Ro*,--
HD 162186,G3IV(e),Ro*,--
BD+21 20A,--,*,solar type dwarf (Nascimbeni 2016)
BD+66 1664,G5,SB*,--
HD 217344,G4V,RS*,--
BD+09 4984B,F2IV/G5V,SB*,"listed as A2 (Anderson 2012, Raghavan 2010)"
V* V369 Gem,G5,RS*,--
HD 146696,G0,SB*,--
HD 156097,G5V,Er*,--


In [30]:
srsc_table = Table.read('data/2026.01.16_12_07_51_SRSC_Static.csv')

In [47]:
solar_like_simbad_ids = solar_like_stars['main_id'].tolist()

In [68]:
# Correct some of the names that were slightly different (missing spaces etc)
solar_like_simbad_ids[28] = 'BD+44 3670'
solar_like_simbad_ids[33] = 'HD 22213'
solar_like_simbad_ids[39] = 'HD 23524'
solar_like_simbad_ids[29] = 'V* V815 Her'

In [66]:
mask = np.isin(srsc_table['Simbad_ID'], solar_like_simbad_ids)
solar_like_coords = srsc_table[mask][['Simbad_ID', 'Archival_ra', 'Archival_dec']]

In [67]:
solar_like_coords

Simbad_ID,Archival_ra,Archival_dec
str31,float64,float64
HD 102982,177.787504285457,-51.876071495961
V* V376 Cep,329.577295376914,82.870902021925
V* V815 Her,272.06734740577,29.6910259723
HD 165590A,271.457229,21.44599
HD 82533,142.67769359508,-53.521472152447
HD 217344,345.11766870906,-33.745748964489
HD 8558,20.839329303534,-57.480907517274
HD 200244,315.742708658278,-22.933914165698
HD 162186,267.778490987213,-32.307805803789


Now that we have a list of our sources and their coordinates, we can do a CASDA query on those coordinates to find out what observations there are for each of these sources. Let's start with a test source and go from there once we know the layout.

In [69]:
ra = solar_like_coords[0]['Archival_ra']
dec = solar_like_coords[0]['Archival_dec']
coords = SkyCoord(ra = ra*u.deg, dec = dec*u.deg, frame='icrs')

In [92]:
result_table = Casda.query_region(coords, radius="5 arcmin")
public_data = Casda.filter_out_unreleased(result_table)

min_exptime = 60*60 # 1 Hour minimum time
dataproduct_type = 'visibility' # We want visibility data

mask = (public_data['dataproduct_type'] == dataproduct_type) & (public_data['t_exptime'] >= min_exptime)
results = public_data[mask]
results

# n_obs = len(np.unique(results['obs_id']))
obs_names = list(np.unique(results['obs_collection']).tolist())
obs_names

['FLASH', 'WALLABY']

Assuming we have done it correctly, this is a way to get the number of unique observations per source. Lets quickly loop this, and print out some more data as well as appending it to our table.

In [98]:
# Add some columns to our table
solar_like_coords['n_obs'] = [0] * len(solar_like_coords)
solar_like_coords['survey_names'] = Column(['N/A']*len(solar_like_coords), dtype='U100')

In [99]:
# Set some global parameters
min_exptime = 60*60 # 1 Hour minimum time
dataproduct_type = 'visibility' # We want visibility data only

for i, row in enumerate(solar_like_coords):

    # Get the coordinates
    ra = row['Archival_ra']
    dec = row['Archival_dec']
    coords = SkyCoord(ra = ra*u.deg, dec = dec*u.deg, frame='icrs')

    # CASDA Query
    result_table = Casda.query_region(coords, radius="5 arcmin")
    public_data = Casda.filter_out_unreleased(result_table)
    mask = (public_data['dataproduct_type'] == dataproduct_type) & (public_data['t_exptime'] >= min_exptime)
    results = public_data[mask]

    # Here is the data we want to append
    n_obs = len(np.unique(results['obs_id']))
    obs_names = list(np.unique(results['obs_collection']).tolist())

    # Append our data to the table
    solar_like_coords[i]['n_obs'] = n_obs
    solar_like_coords[i]['survey_names'] = ', '.join(obs_names)

In [102]:
total_obs = solar_like_coords['n_obs'].sum()
total_obs

np.int64(80)

In [109]:
solar_like_coords

Simbad_ID,Archival_ra,Archival_dec,n_obs,survey_names
str31,float64,float64,int64,str100
HD 102982,177.787504285457,-51.876071495961,2,"FLASH, WALLABY"
V* V376 Cep,329.577295376914,82.870902021925,0,
V* V815 Her,272.06734740577,29.6910259723,0,
HD 165590A,271.457229,21.44599,0,
HD 82533,142.67769359508,-53.521472152447,3,"EMU, GASKAP-HI"
HD 217344,345.11766870906,-33.745748964489,22,"DINGO, FLASH, WALLABY"
HD 8558,20.839329303534,-57.480907517274,2,WALLABY
HD 200244,315.742708658278,-22.933914165698,2,FLASH
HD 162186,267.778490987213,-32.307805803789,10,An ASKAP Survey for Galactic Radio Transients in the Baade's Window
